In [ ]:
# @title
#XGBoost v3_auc
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

# =========================
# 1. 讀資料
# =========================
train_path = "train.csv"
test_path = "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# target
target_col = "Bankrupt"

X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].copy()
X_test = test_df.copy()

print("Train shape:", X.shape)
print("Test shape :", X_test.shape)
print("Positive ratio:", y.mean())

# =========================
# 2. 缺值處理
#    - median 補值
#    - 額外加 missing indicator
# =========================
# missing indicator
X_missing = X.isna().astype(int)
X_missing.columns = [f"{c}_isna" for c in X_missing.columns]

X_test_missing = X_test.isna().astype(int)
X_test_missing.columns = [f"{c}_isna" for c in X_test_missing.columns]

# combine original + missing indicator
X_full = pd.concat([X, X_missing], axis=1)
X_test_full = pd.concat([X_test, X_test_missing], axis=1)

# impute
imputer = SimpleImputer(strategy="median")
X_full_imputed = pd.DataFrame(
    imputer.fit_transform(X_full),
    columns=X_full.columns
)
X_test_full_imputed = pd.DataFrame(
    imputer.transform(X_test_full),
    columns=X_test_full.columns
)

# =========================
# 3. Cross-validation
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X_full_imputed))
test_pred_folds = []

# class imbalance
pos = (y == 1).sum()
neg = (y == 0).sum()
scale_pos_weight = neg / pos
print("scale_pos_weight =", scale_pos_weight)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full_imputed, y), 1):
    X_tr = X_full_imputed.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    X_va = X_full_imputed.iloc[va_idx]
    y_va = y.iloc[va_idx]

    model = XGBClassifier(
      n_estimators=1200,
      max_depth=3,
      learning_rate=0.025,
      subsample=0.9,
      colsample_bytree=0.75,
      min_child_weight=5,
      reg_alpha=0.5,
      reg_lambda=2.0,
      objective="binary:logistic",
      eval_metric="aucpr",
      random_state=42 + fold,
      scale_pos_weight=neg / pos,
      n_jobs=-1
  )

    model.fit(X_tr, y_tr)

    va_prob = model.predict_proba(X_va)[:, 1]
    te_prob = model.predict_proba(X_test_full_imputed)[:, 1]

    oof_pred[va_idx] = va_prob
    test_pred_folds.append(te_prob)

    # 先用 0.5 看 fold 表現
    va_pred_05 = (va_prob >= 0.5).astype(int)
    fold_f1 = f1_score(y_va, va_pred_05, average="macro")
    print(f"Fold {fold} Macro-F1 @0.5 = {fold_f1:.6f}")

# =========================
# 4. 找最佳 threshold
#    因為作業評分是 Macro-F1
# =========================
best_th = 0.5
best_f1 = -1

for th in np.arange(0.05, 0.96, 0.01):
    pred = (oof_pred >= th).astype(int)
    score = f1_score(y, pred, average="macro")
    if score > best_f1:
        best_f1 = score
        best_th = th

print(f"Best threshold = {best_th:.2f}")
print(f"OOF Macro-F1   = {best_f1:.6f}")

# =========================
# 5. 產生 test prediction
# =========================
test_prob = np.mean(test_pred_folds, axis=0)
test_label = (test_prob >= best_th).astype(int)

# =========================
# 6. 輸出 submission.csv
#    必須只有 Id, Bankrupt 兩欄
# =========================
submission = pd.DataFrame({
    "Id": np.arange(len(test_label)),
    "Bankrupt": test_label
})

submission.to_csv("v3-auc.csv", index=False)
print("Saved submission.csv")
print(submission.head())

Train shape: (5543, 95)
Test shape : (1386, 95)
Positive ratio: 0.04762763846292621
scale_pos_weight = 19.99621212121212
Fold 1 Macro-F1 @0.5 = 0.752280
Fold 2 Macro-F1 @0.5 = 0.818339
Fold 3 Macro-F1 @0.5 = 0.767280
Fold 4 Macro-F1 @0.5 = 0.788550
Fold 5 Macro-F1 @0.5 = 0.750313
Best threshold = 0.51
OOF Macro-F1   = 0.777867
Saved submission.csv
   Id  Bankrupt
0   0         0
1   1         0
2   2         0
3   3         0
4   4         0


In [ ]:
# @title
# ensemble_hgb_xgb v_hgb
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
WEIGHT_STEP = 0.05

def search_best_threshold(y_true, y_prob, start=0.05, end=0.95, step=0.01):
    best_t = 0.5
    best_f1 = -1.0
    for t in np.arange(start, end, step):
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, average="macro")
        if score > best_f1:
            best_f1 = score
            best_t = t
    return best_t, best_f1

def save_submission_from_prob(test_prob, threshold, filename):
    pred = (test_prob >= threshold).astype(int)
    submission = pd.DataFrame({
        "Id": np.arange(len(pred)),
        "Bankrupt": pred
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}")
    print(submission.head())
    return submission

def search_best_ensemble_weights(
    y_true,
    oof_dict,
    model_names,
    weight_step=0.05,
    threshold_start=0.05,
    threshold_end=0.95,
    threshold_step=0.01
):
    m1, m2 = model_names
    best_score = -1.0
    best_threshold = None
    best_weights_dict = None

    for w1 in np.arange(0.0, 1.0 + 1e-9, weight_step):
        w2 = 1.0 - w1
        oof_pred = w1 * oof_dict[m1] + w2 * oof_dict[m2]
        t, score = search_best_threshold(
            y_true, oof_pred,
            start=threshold_start,
            end=threshold_end,
            step=threshold_step
        )

        if score > best_score:
            best_score = score
            best_threshold = t
            best_weights_dict = {
                m1: round(float(w1), 4),
                m2: round(float(w2), 4)
            }

    return best_weights_dict, best_threshold, best_score

def build_ensemble_prediction(pred_dict, weights_dict):
    output = np.zeros(len(next(iter(pred_dict.values()))))
    for model_name, weight in weights_dict.items():
        output += weight * pred_dict[model_name]
    return output

# 1. load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

X = train_df.drop(columns=["Bankrupt"])
y = train_df["Bankrupt"].astype(int)
X_test = test_df.copy()

print("Train shape:", X.shape)
print("Test shape :", X_test.shape)
print("Positive ratio:", y.mean())

# 2. impute
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 3. model config
pos_ratio = y.mean()
neg_ratio = 1.0 - pos_ratio
scale_pos_weight = neg_ratio / pos_ratio

models = {
    "hgb": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_depth=6,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    ),
    "xgb": XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=2.0,
        min_child_weight=2,
        gamma=0.1,
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist"
    )
}

# 4. cv
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_pred_each_model = {name: np.zeros(len(X_imp)) for name in models}
test_pred_each_model = {name: np.zeros(len(X_test_imp)) for name in models}
single_model_results = {}

for model_name, model in models.items():
    print(f"\n===== Training {model_name} =====")
    fold_scores = []
    fold_thresholds = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
        X_tr, X_va = X_imp.iloc[tr_idx], X_imp.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        pos_weight_fold = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)
        sample_weight = np.where(y_tr == 1, pos_weight_fold, 1.0)

        if model_name == "hgb":
            model.fit(X_tr, y_tr, sample_weight=sample_weight)
        else:
            model.fit(X_tr, y_tr)

        va_prob = model.predict_proba(X_va)[:, 1]
        te_prob = model.predict_proba(X_test_imp)[:, 1]

        oof_pred_each_model[model_name][va_idx] = va_prob
        test_pred_each_model[model_name] += te_prob / N_SPLITS

        best_t, best_f1 = search_best_threshold(y_va, va_prob)
        fold_thresholds.append(best_t)
        fold_scores.append(best_f1)

        print(f"Fold {fold}: best threshold = {best_t:.2f}, macro-F1 = {best_f1:.5f}")

    overall_t, overall_f1 = search_best_threshold(y, oof_pred_each_model[model_name])

    single_model_results[model_name] = {
        "fold_thresholds": fold_thresholds,
        "fold_scores": fold_scores,
        "oof_best_threshold": overall_t,
        "oof_macro_f1": overall_f1,
        "mean_cv_macro_f1": float(np.mean(fold_scores))
    }

    print(f"[{model_name}] OOF best threshold = {overall_t:.2f}, OOF macro-F1 = {overall_f1:.5f}")
    print(f"[{model_name}] mean CV macro-F1 = {np.mean(fold_scores):.5f}")

# 5. save single model
print("\n===== Saving single-model submissions =====")
for model_name in models.keys():
    best_t = single_model_results[model_name]["oof_best_threshold"]
    best_f1 = single_model_results[model_name]["oof_macro_f1"]

    print(f"{model_name}: OOF best threshold = {best_t:.2f}, OOF macro-F1 = {best_f1:.5f}")
    save_submission_from_prob(
        test_prob=test_pred_each_model[model_name],
        threshold=best_t,
        filename=f"v_{model_name}.csv"
    )

# 6. search best hgb + xgb ensemble
print("\n===== Searching best ensemble weights (hgb + xgb) =====")
best_weights, best_threshold, best_score = search_best_ensemble_weights(
    y_true=y,
    oof_dict=oof_pred_each_model,
    model_names=["hgb", "xgb"],
    weight_step=WEIGHT_STEP,
    threshold_start=0.05,
    threshold_end=0.95,
    threshold_step=0.01
)

print("Best weights:", best_weights)
print("Best threshold:", round(best_threshold, 4))
print("Best OOF Macro-F1:", round(best_score, 5))

test_ensemble = build_ensemble_prediction(test_pred_each_model, best_weights)
save_submission_from_prob(
    test_prob=test_ensemble,
    threshold=best_threshold,
    filename="ensemble_hgb_xgb.csv"
)

# 7. summary
print("\n===== Summary =====")
for model_name, result in single_model_results.items():
    print(
        f"{model_name:>4} | "
        f"OOF Macro-F1 = {result['oof_macro_f1']:.5f} | "
        f"Threshold = {result['oof_best_threshold']:.2f}"
    )

print(
    f" ens2 | OOF Macro-F1 = {best_score:.5f} | "
    f"Threshold = {best_threshold:.2f} | "
    f"Weights = {best_weights}"
)

print("\nAll done.")

Train shape: (5543, 95)
Test shape : (1386, 95)
Positive ratio: 0.04762763846292621

===== Training hgb =====
Fold 1: best threshold = 0.11, macro-F1 = 0.76454
Fold 2: best threshold = 0.60, macro-F1 = 0.83986
Fold 3: best threshold = 0.79, macro-F1 = 0.78858
Fold 4: best threshold = 0.29, macro-F1 = 0.83626
Fold 5: best threshold = 0.07, macro-F1 = 0.80427
[hgb] OOF best threshold = 0.26, OOF macro-F1 = 0.78574
[hgb] mean CV macro-F1 = 0.80670

===== Training xgb =====
Fold 1: best threshold = 0.51, macro-F1 = 0.77448
Fold 2: best threshold = 0.76, macro-F1 = 0.83336
Fold 3: best threshold = 0.87, macro-F1 = 0.79348
Fold 4: best threshold = 0.68, macro-F1 = 0.83504
Fold 5: best threshold = 0.38, macro-F1 = 0.78644
[xgb] OOF best threshold = 0.51, OOF macro-F1 = 0.78309
[xgb] mean CV macro-F1 = 0.80456

===== Saving single-model submissions =====
hgb: OOF best threshold = 0.26, OOF macro-F1 = 0.78574
Saved v_hgb.csv
   Id  Bankrupt
0   0         0
1   1         0
2   2         0
3   3 

In [ ]:
# @title
#small sweeps
import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =========================================================
# 0. Config
# =========================================================
RANDOM_STATE = 42
N_SPLITS = 5

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

# =========================================================
# 1. Helper functions
# =========================================================
def search_best_threshold(y_true, y_prob, start=0.05, end=0.95, step=0.01):
    best_t = 0.5
    best_f1 = -1.0

    for t in np.arange(start, end, step):
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, average="macro")
        if score > best_f1:
            best_f1 = score
            best_t = t

    return best_t, best_f1


def save_submission_from_prob(test_prob, threshold, filename):
    pred = (test_prob >= threshold).astype(int)
    submission = pd.DataFrame({
        "Id": np.arange(len(pred)),
        "Bankrupt": pred
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}")
    return submission


def build_hgb(params):
    return HistGradientBoostingClassifier(
        learning_rate=params["learning_rate"],
        max_iter=params["max_iter"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        l2_regularization=params["l2_regularization"],
        random_state=RANDOM_STATE
    )


def build_xgb(params, scale_pos_weight):
    return XGBClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        min_child_weight=params["min_child_weight"],
        gamma=params["gamma"],
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        early_stopping_rounds=100
    )


def evaluate_model_cv(
    model_type,
    params,
    X_imp,
    y,
    X_test_imp,
    skf,
    scale_pos_weight
):
    oof_pred = np.zeros(len(X_imp))
    test_pred = np.zeros(len(X_test_imp))
    fold_scores = []
    fold_thresholds = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
        X_tr = X_imp.iloc[tr_idx]
        X_va = X_imp.iloc[va_idx]
        y_tr = y.iloc[tr_idx]
        y_va = y.iloc[va_idx]

        # fold-specific positive weight
        pos_weight_fold = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)
        sample_weight = np.where(y_tr == 1, pos_weight_fold, 1.0)

        if model_type == "hgb":
            model = build_hgb(params)
            model.fit(X_tr, y_tr, sample_weight=sample_weight)
        elif model_type == "xgb":
            model = build_xgb(params, scale_pos_weight=scale_pos_weight)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_va, y_va)],
                verbose=False
            )
        else:
            raise ValueError("Unsupported model_type")

        va_prob = model.predict_proba(X_va)[:, 1]
        te_prob = model.predict_proba(X_test_imp)[:, 1]

        oof_pred[va_idx] = va_prob
        test_pred += te_prob / N_SPLITS

        fold_t, fold_f1 = search_best_threshold(y_va, va_prob)
        fold_thresholds.append(fold_t)
        fold_scores.append(fold_f1)

    overall_t, overall_f1 = search_best_threshold(y, oof_pred)

    return {
        "params": params,
        "oof_pred": oof_pred,
        "test_pred": test_pred,
        "fold_scores": fold_scores,
        "fold_thresholds": fold_thresholds,
        "mean_cv_macro_f1": float(np.mean(fold_scores)),
        "oof_best_threshold": float(overall_t),
        "oof_macro_f1": float(overall_f1)
    }


# =========================================================
# 2. Load data
# =========================================================
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

X = train_df.drop(columns=["Bankrupt"])
y = train_df["Bankrupt"].astype(int)
X_test = test_df.copy()

print("Train shape:", X.shape)
print("Test shape :", X_test.shape)
print("Positive ratio:", y.mean())

# =========================================================
# 3. Imputation
# =========================================================
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# =========================================================
# 4. CV and imbalance config
# =========================================================
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

pos_ratio = y.mean()
neg_ratio = 1.0 - pos_ratio
scale_pos_weight = neg_ratio / pos_ratio

print("scale_pos_weight:", scale_pos_weight)

# =========================================================
# 5. Small sweep grids (~12 each)
# =========================================================
hgb_grid = [
    {"learning_rate": 0.03, "max_iter": 300, "max_depth": 4, "min_samples_leaf": 20, "l2_regularization": 0.5},
    {"learning_rate": 0.03, "max_iter": 400, "max_depth": 4, "min_samples_leaf": 30, "l2_regularization": 1.0},
    {"learning_rate": 0.03, "max_iter": 500, "max_depth": 5, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.03, "max_iter": 500, "max_depth": 5, "min_samples_leaf": 30, "l2_regularization": 2.0},
    {"learning_rate": 0.05, "max_iter": 250, "max_depth": 4, "min_samples_leaf": 20, "l2_regularization": 0.5},
    {"learning_rate": 0.05, "max_iter": 300, "max_depth": 5, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 300, "max_depth": 6, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 400, "max_depth": 5, "min_samples_leaf": 30, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 400, "max_depth": 6, "min_samples_leaf": 30, "l2_regularization": 2.0},
    {"learning_rate": 0.07, "max_iter": 200, "max_depth": 4, "min_samples_leaf": 20, "l2_regularization": 0.5},
    {"learning_rate": 0.07, "max_iter": 250, "max_depth": 5, "min_samples_leaf": 20, "l2_regularization": 1.0},
    {"learning_rate": 0.07, "max_iter": 300, "max_depth": 5, "min_samples_leaf": 30, "l2_regularization": 2.0},
]

xgb_grid = [
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.5, "reg_lambda": 2.0, "min_child_weight": 2, "gamma": 0.0},
    {"n_estimators": 800, "max_depth": 3, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 1.0, "reg_lambda": 3.0, "min_child_weight": 3, "gamma": 0.2},
    {"n_estimators": 1000, "max_depth": 3, "learning_rate": 0.02, "subsample": 0.85, "colsample_bytree": 0.80, "reg_alpha": 1.0, "reg_lambda": 3.0, "min_child_weight": 3, "gamma": 0.2},
    {"n_estimators": 600, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.5, "reg_lambda": 2.0, "min_child_weight": 2, "gamma": 0.1},
    {"n_estimators": 800, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.5, "reg_lambda": 2.0, "min_child_weight": 2, "gamma": 0.1},
    {"n_estimators": 1000, "max_depth": 4, "learning_rate": 0.02, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 1.0, "reg_lambda": 3.0, "min_child_weight": 3, "gamma": 0.2},
    {"n_estimators": 500, "max_depth": 5, "learning_rate": 0.03, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.5, "reg_lambda": 2.0, "min_child_weight": 2, "gamma": 0.1},
    {"n_estimators": 700, "max_depth": 5, "learning_rate": 0.02, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 1.0, "reg_lambda": 3.0, "min_child_weight": 3, "gamma": 0.2},
    {"n_estimators": 900, "max_depth": 3, "learning_rate": 0.015, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 1.5, "reg_lambda": 4.0, "min_child_weight": 4, "gamma": 0.2},
    {"n_estimators": 1200, "max_depth": 3, "learning_rate": 0.015, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 1.5, "reg_lambda": 4.0, "min_child_weight": 4, "gamma": 0.3},
    {"n_estimators": 700, "max_depth": 4, "learning_rate": 0.02, "subsample": 0.75, "colsample_bytree": 0.80, "reg_alpha": 1.0, "reg_lambda": 3.0, "min_child_weight": 3, "gamma": 0.2},
    {"n_estimators": 900, "max_depth": 4, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.75, "reg_alpha": 1.0, "reg_lambda": 3.0, "min_child_weight": 3, "gamma": 0.3},
]

# =========================================================
# 6. Run HGB sweep
# =========================================================
hgb_results = []

print("\n================ HGB SWEEP ================")
for i, params in enumerate(hgb_grid, 1):
    print(f"\n[HGB {i}/{len(hgb_grid)}] params = {params}")

    result = evaluate_model_cv(
        model_type="hgb",
        params=params,
        X_imp=X_imp,
        y=y,
        X_test_imp=X_test_imp,
        skf=skf,
        scale_pos_weight=scale_pos_weight
    )

    row = {
        "rank_tmp": i,
        "model": "hgb",
        "params": str(params),
        "oof_macro_f1": result["oof_macro_f1"],
        "oof_best_threshold": result["oof_best_threshold"],
        "mean_cv_macro_f1": result["mean_cv_macro_f1"],
        "fold_scores": str([round(x, 5) for x in result["fold_scores"]]),
        "fold_thresholds": str([round(x, 2) for x in result["fold_thresholds"]]),
        "result_obj": result
    }
    hgb_results.append(row)

    print(
        f"OOF Macro-F1 = {result['oof_macro_f1']:.5f} | "
        f"Threshold = {result['oof_best_threshold']:.2f} | "
        f"Mean CV = {result['mean_cv_macro_f1']:.5f}"
    )

hgb_results_df = pd.DataFrame(hgb_results).sort_values(
    by="oof_macro_f1", ascending=False
).reset_index(drop=True)

hgb_results_df["final_rank"] = np.arange(1, len(hgb_results_df) + 1)
hgb_results_df.to_csv("hgb_sweep_results.csv", index=False)
print("\nSaved hgb_sweep_results.csv")

# save top 3 HGB submissions
print("\nTop 3 HGB:")
for i in range(min(3, len(hgb_results_df))):
    row = hgb_results_df.iloc[i]
    result = row["result_obj"]
    filename = f"v_hgb_sweep{i+1}.csv"

    print(
        f"Rank {i+1} | OOF = {row['oof_macro_f1']:.5f} | "
        f"Threshold = {row['oof_best_threshold']:.2f} | Params = {row['params']}"
    )

    save_submission_from_prob(
        test_prob=result["test_pred"],
        threshold=result["oof_best_threshold"],
        filename=filename
    )

# =========================================================
# 7. Run XGB sweep
# =========================================================
xgb_results = []

print("\n================ XGB SWEEP ================")
for i, params in enumerate(xgb_grid, 1):
    print(f"\n[XGB {i}/{len(xgb_grid)}] params = {params}")

    result = evaluate_model_cv(
        model_type="xgb",
        params=params,
        X_imp=X_imp,
        y=y,
        X_test_imp=X_test_imp,
        skf=skf,
        scale_pos_weight=scale_pos_weight
    )

    row = {
        "rank_tmp": i,
        "model": "xgb",
        "params": str(params),
        "oof_macro_f1": result["oof_macro_f1"],
        "oof_best_threshold": result["oof_best_threshold"],
        "mean_cv_macro_f1": result["mean_cv_macro_f1"],
        "fold_scores": str([round(x, 5) for x in result["fold_scores"]]),
        "fold_thresholds": str([round(x, 2) for x in result["fold_thresholds"]]),
        "result_obj": result
    }
    xgb_results.append(row)

    print(
        f"OOF Macro-F1 = {result['oof_macro_f1']:.5f} | "
        f"Threshold = {result['oof_best_threshold']:.2f} | "
        f"Mean CV = {result['mean_cv_macro_f1']:.5f}"
    )

xgb_results_df = pd.DataFrame(xgb_results).sort_values(
    by="oof_macro_f1", ascending=False
).reset_index(drop=True)

xgb_results_df["final_rank"] = np.arange(1, len(xgb_results_df) + 1)
xgb_results_df.to_csv("xgb_sweep_results.csv", index=False)
print("\nSaved xgb_sweep_results.csv")

# save top 3 XGB submissions
print("\nTop 3 XGB:")
for i in range(min(3, len(xgb_results_df))):
    row = xgb_results_df.iloc[i]
    result = row["result_obj"]
    filename = f"v_xgb_sweep{i+1}.csv"

    print(
        f"Rank {i+1} | OOF = {row['oof_macro_f1']:.5f} | "
        f"Threshold = {row['oof_best_threshold']:.2f} | Params = {row['params']}"
    )

    save_submission_from_prob(
        test_prob=result["test_pred"],
        threshold=result["oof_best_threshold"],
        filename=filename
    )

# =========================================================
# 8. Compact summary
# =========================================================
print("\n================ FINAL SUMMARY ================")

print("\nTop 3 HGB")
print(hgb_results_df[[
    "final_rank", "oof_macro_f1", "oof_best_threshold", "mean_cv_macro_f1", "params"
]].head(3).to_string(index=False))

print("\nTop 3 XGB")
print(xgb_results_df[[
    "final_rank", "oof_macro_f1", "oof_best_threshold", "mean_cv_macro_f1", "params"
]].head(3).to_string(index=False))

print("\nDone.")

Train shape: (5543, 95)
Test shape : (1386, 95)
Positive ratio: 0.04762763846292621
scale_pos_weight: 19.996212121212125

================ HGB SWEEP ================

[HGB 1/12] params = {'learning_rate': 0.03, 'max_iter': 300, 'max_depth': 4, 'min_samples_leaf': 20, 'l2_regularization': 0.5}
OOF Macro-F1 = 0.78142 | Threshold = 0.59 | Mean CV = 0.79208

[HGB 2/12] params = {'learning_rate': 0.03, 'max_iter': 400, 'max_depth': 4, 'min_samples_leaf': 30, 'l2_regularization': 1.0}
OOF Macro-F1 = 0.78448 | Threshold = 0.77 | Mean CV = 0.80482

[HGB 3/12] params = {'learning_rate': 0.03, 'max_iter': 500, 'max_depth': 5, 'min_samples_leaf': 20, 'l2_regularization': 1.0}
OOF Macro-F1 = 0.78339 | Threshold = 0.68 | Mean CV = 0.80393

[HGB 4/12] params = {'learning_rate': 0.03, 'max_iter': 500, 'max_depth': 5, 'min_samples_leaf': 30, 'l2_regularization': 2.0}
OOF Macro-F1 = 0.78711 | Threshold = 0.74 | Mean CV = 0.80844

[HGB 5/12] params = {'learning_rate': 0.05, 'max_iter': 250, 'max_depth':

In [ ]:
# @title
# ensemble_hgb_xgb_2
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
WEIGHT_STEP = 0.02

def search_best_threshold(y_true, y_prob, start=0.05, end=0.95, step=0.01):
    best_t = 0.5
    best_f1 = -1.0
    for t in np.arange(start, end, step):
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, average="macro")
        if score > best_f1:
            best_f1 = score
            best_t = t
    return best_t, best_f1

def save_submission_from_prob(test_prob, threshold, filename):
    pred = (test_prob >= threshold).astype(int)
    submission = pd.DataFrame({
        "Id": np.arange(len(pred)),
        "Bankrupt": pred
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}")
    print(submission.head())
    return submission

def search_best_ensemble_weights(
    y_true,
    oof_dict,
    model_names,
    weight_step=0.02,
    threshold_start=0.1,
    threshold_end=0.9,
    threshold_step=0.01
):
    m1, m2 = model_names
    best_score = -1.0
    best_threshold = None
    best_weights_dict = None

    for w1 in np.arange(0.0, 1.0 + 1e-9, weight_step):
        w2 = 1.0 - w1
        oof_pred = w1 * oof_dict[m1] + w2 * oof_dict[m2]
        t, score = search_best_threshold(
            y_true, oof_pred,
            start=threshold_start,
            end=threshold_end,
            step=threshold_step
        )

        if score > best_score:
            best_score = score
            best_threshold = t
            best_weights_dict = {
                m1: round(float(w1), 4),
                m2: round(float(w2), 4)
            }

    return best_weights_dict, best_threshold, best_score

def build_ensemble_prediction(pred_dict, weights_dict):
    output = np.zeros(len(next(iter(pred_dict.values()))))
    for model_name, weight in weights_dict.items():
        output += weight * pred_dict[model_name]
    return output

# 1. load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

X = train_df.drop(columns=["Bankrupt"])
y = train_df["Bankrupt"].astype(int)
X_test = test_df.copy()

print("Train shape:", X.shape)
print("Test shape :", X_test.shape)
print("Positive ratio:", y.mean())

# 2. impute
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 3. model config
pos_ratio = y.mean()
neg_ratio = 1.0 - pos_ratio
scale_pos_weight = neg_ratio / pos_ratio

models = {
    "hgb": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_depth=6,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    ),
    "xgb": XGBClassifier(
      n_estimators=600,
      max_depth=4,
      learning_rate=0.03,
      subsample=0.8,
      colsample_bytree=0.8,
      reg_alpha=0.5,
      reg_lambda=2.0,
      min_child_weight=2,
      gamma=0.1,
      objective="binary:logistic",
      eval_metric="aucpr",
      scale_pos_weight=scale_pos_weight,
      random_state=RANDOM_STATE,
      n_jobs=-1,
      tree_method="hist"
  )
}

# 4. cv
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_pred_each_model = {name: np.zeros(len(X_imp)) for name in models}
test_pred_each_model = {name: np.zeros(len(X_test_imp)) for name in models}
single_model_results = {}

for model_name, model in models.items():
    print(f"\n===== Training {model_name} =====")
    fold_scores = []
    fold_thresholds = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
        X_tr, X_va = X_imp.iloc[tr_idx], X_imp.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        pos_weight_fold = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)
        sample_weight = np.where(y_tr == 1, pos_weight_fold, 1.0)

        if model_name == "hgb":
            model.fit(X_tr, y_tr, sample_weight=sample_weight)
        else:
            model.fit(X_tr, y_tr)

        va_prob = model.predict_proba(X_va)[:, 1]
        te_prob = model.predict_proba(X_test_imp)[:, 1]

        oof_pred_each_model[model_name][va_idx] = va_prob
        test_pred_each_model[model_name] += te_prob / N_SPLITS

        best_t, best_f1 = search_best_threshold(y_va, va_prob)
        fold_thresholds.append(best_t)
        fold_scores.append(best_f1)

        print(f"Fold {fold}: best threshold = {best_t:.2f}, macro-F1 = {best_f1:.5f}")

    overall_t, overall_f1 = search_best_threshold(y, oof_pred_each_model[model_name])

    single_model_results[model_name] = {
        "fold_thresholds": fold_thresholds,
        "fold_scores": fold_scores,
        "oof_best_threshold": overall_t,
        "oof_macro_f1": overall_f1,
        "mean_cv_macro_f1": float(np.mean(fold_scores))
    }

    print(f"[{model_name}] OOF best threshold = {overall_t:.2f}, OOF macro-F1 = {overall_f1:.5f}")
    print(f"[{model_name}] mean CV macro-F1 = {np.mean(fold_scores):.5f}")

# 5. save single model
print("\n===== Saving single-model submissions =====")
for model_name in models.keys():
    best_t = single_model_results[model_name]["oof_best_threshold"]
    best_f1 = single_model_results[model_name]["oof_macro_f1"]

    print(f"{model_name}: OOF best threshold = {best_t:.2f}, OOF macro-F1 = {best_f1:.5f}")
    save_submission_from_prob(
        test_prob=test_pred_each_model[model_name],
        threshold=best_t,
        filename=f"v_{model_name}.csv"
    )

# 6. search best hgb + xgb ensemble
print("\n===== Searching best ensemble weights (hgb + xgb) =====")
best_weights, best_threshold, best_score = search_best_ensemble_weights(
    y_true=y,
    oof_dict=oof_pred_each_model,
    model_names=["hgb", "xgb"],
    weight_step=WEIGHT_STEP,
    threshold_start=0.05,
    threshold_end=0.95,
    threshold_step=0.01
)

print("Best weights:", best_weights)
print("Best threshold:", round(best_threshold, 4))
print("Best OOF Macro-F1:", round(best_score, 5))

test_ensemble = build_ensemble_prediction(test_pred_each_model, best_weights)
save_submission_from_prob(
    test_prob=test_ensemble,
    threshold=best_threshold,
    filename="ensemble_hgb_xgb_2.csv"
)

# 7. summary
print("\n===== Summary =====")
for model_name, result in single_model_results.items():
    print(
        f"{model_name:>4} | "
        f"OOF Macro-F1 = {result['oof_macro_f1']:.5f} | "
        f"Threshold = {result['oof_best_threshold']:.2f}"
    )

print(
    f" ens2 | OOF Macro-F1 = {best_score:.5f} | "
    f"Threshold = {best_threshold:.2f} | "
    f"Weights = {best_weights}"
)

# 8. Threshold fine sweep around current ensemble threshold
oof_ensemble = build_ensemble_prediction(oof_pred_each_model, best_weights)
threshold_candidates = [0.35, 0.37, 0.39, 0.40, 0.41, 0.42, 0.43, 0.45, 0.47, 0.49]

rows = []
for t in threshold_candidates:
    score = f1_score(y, (oof_ensemble >= t).astype(int), average="macro")
    rows.append((t, score))

df = pd.DataFrame(rows, columns=["threshold", "oof_macro_f1"])
df = df.sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
print(df)

for i in range(3):
    t = float(df.loc[i, "threshold"])
    pred = (test_ensemble >= t).astype(int)
    pd.DataFrame({
        "Id": np.arange(len(pred)),
        "Bankrupt": pred
    }).to_csv(f"ensemble_hgb_xgb_tune{i+1}.csv", index=False)
    print(f"Saved ensemble_hgb_xgb_tune{i+1}.csv with threshold {t}")
print("\nAll done.")

Train shape: (5543, 95)
Test shape : (1386, 95)
Positive ratio: 0.04762763846292621

===== Training hgb =====
Fold 1: best threshold = 0.11, macro-F1 = 0.76454
Fold 2: best threshold = 0.60, macro-F1 = 0.83986
Fold 3: best threshold = 0.79, macro-F1 = 0.78858
Fold 4: best threshold = 0.29, macro-F1 = 0.83626
Fold 5: best threshold = 0.07, macro-F1 = 0.80427
[hgb] OOF best threshold = 0.26, OOF macro-F1 = 0.78574
[hgb] mean CV macro-F1 = 0.80670

===== Training xgb =====
Fold 1: best threshold = 0.50, macro-F1 = 0.78245
Fold 2: best threshold = 0.64, macro-F1 = 0.82919
Fold 3: best threshold = 0.84, macro-F1 = 0.78965
Fold 4: best threshold = 0.75, macro-F1 = 0.82579
Fold 5: best threshold = 0.16, macro-F1 = 0.77397
[xgb] OOF best threshold = 0.51, OOF macro-F1 = 0.78127
[xgb] mean CV macro-F1 = 0.80021

===== Saving single-model submissions =====
hgb: OOF best threshold = 0.26, OOF macro-F1 = 0.78574
Saved v_hgb.csv
   Id  Bankrupt
0   0         0
1   1         0
2   2         0
3   3 

In [ ]:
# add 1. two-stage threshold tuning (smaller step) 2. prob/rank-based ensemble 3. quantile clipping
# all clips output comes from
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 5
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
WEIGHT_STEP = 0.02


# =========================
# Utility functions
# =========================
def search_best_threshold_two_stage(
    y_true,
    y_prob,
    coarse_start=0.01,
    coarse_end=0.99,
    coarse_step=0.01,
    fine_window=0.03,
    fine_step=0.001
):
    best_t = 0.5
    best_f1 = -1.0

    # coarse search
    for t in np.arange(coarse_start, coarse_end + 1e-12, coarse_step):
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, average="macro")
        if score > best_f1:
            best_f1 = score
            best_t = t

    # fine search
    fine_start = max(0.0, best_t - fine_window)
    fine_end = min(1.0, best_t + fine_window)

    for t in np.arange(fine_start, fine_end + 1e-12, fine_step):
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, average="macro")
        if score > best_f1:
            best_f1 = score
            best_t = t

    return float(best_t), float(best_f1)


def save_submission_from_prob(test_prob, threshold, filename):
    pred = (test_prob >= threshold).astype(int)
    submission = pd.DataFrame({
        "Id": np.arange(len(pred)),
        "Bankrupt": pred
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}")
    print(submission.head())
    return submission


def clip_by_quantile(train_df, test_df, low=0.005, high=0.995):
    train_clip = train_df.copy()
    test_clip = test_df.copy()

    lower_bounds = train_df.quantile(low)
    upper_bounds = train_df.quantile(high)

    train_clip = train_df.clip(lower=lower_bounds, upper=upper_bounds, axis=1)
    test_clip = test_df.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

    return train_clip, test_clip


def to_rank(arr):
    return pd.Series(arr).rank(method="average", pct=True).to_numpy()


def build_ensemble_prediction(pred_dict, weights_dict):
    output = np.zeros(len(next(iter(pred_dict.values()))))
    for model_name, weight in weights_dict.items():
        output += weight * pred_dict[model_name]
    return output


def build_rank_ensemble_prediction(pred_dict, weights_dict):
    output = np.zeros(len(next(iter(pred_dict.values()))))
    for model_name, weight in weights_dict.items():
        output += weight * to_rank(pred_dict[model_name])
    return output


def search_best_ensemble_weights(
    y_true,
    oof_dict,
    model_names,
    weight_step=0.02
):
    m1, m2 = model_names
    best_score = -1.0
    best_threshold = None
    best_weights_dict = None

    for w1 in np.arange(0.0, 1.0 + 1e-9, weight_step):
        w2 = 1.0 - w1
        oof_pred = w1 * oof_dict[m1] + w2 * oof_dict[m2]

        t, score = search_best_threshold_two_stage(
            y_true=y_true,
            y_prob=oof_pred
        )

        if score > best_score:
            best_score = score
            best_threshold = t
            best_weights_dict = {
                m1: round(float(w1), 4),
                m2: round(float(w2), 4)
            }

    return best_weights_dict, best_threshold, best_score


def search_best_rank_ensemble_weights(
    y_true,
    oof_dict,
    model_names,
    weight_step=0.02
):
    m1, m2 = model_names
    best_score = -1.0
    best_threshold = None
    best_weights_dict = None

    r1 = to_rank(oof_dict[m1])
    r2 = to_rank(oof_dict[m2])

    for w1 in np.arange(0.0, 1.0 + 1e-9, weight_step):
        w2 = 1.0 - w1
        oof_rank_pred = w1 * r1 + w2 * r2

        t, score = search_best_threshold_two_stage(
            y_true=y_true,
            y_prob=oof_rank_pred
        )

        if score > best_score:
            best_score = score
            best_threshold = t
            best_weights_dict = {
                m1: round(float(w1), 4),
                m2: round(float(w2), 4)
            }

    return best_weights_dict, best_threshold, best_score


def run_pipeline(
    clip_low=0.005,
    clip_high=0.995,
    output_prefix="clip005_995"
):
    print("=" * 80)
    print(f"Running pipeline with clip_low={clip_low}, clip_high={clip_high}")
    print("=" * 80)

    # =========================
    # 1. load data
    # =========================
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)

    X = train_df.drop(columns=["Bankrupt"])
    y = train_df["Bankrupt"].astype(int)
    X_test = test_df.copy()

    print("Train shape:", X.shape)
    print("Test shape :", X_test.shape)
    print("Positive ratio:", y.mean())

    # =========================
    # 2. clipping + impute
    # =========================
    X_clip, X_test_clip = clip_by_quantile(X, X_test, low=clip_low, high=clip_high)

    imputer = SimpleImputer(strategy="median")
    X_imp = pd.DataFrame(imputer.fit_transform(X_clip), columns=X.columns)
    X_test_imp = pd.DataFrame(imputer.transform(X_test_clip), columns=X_test.columns)

    # =========================
    # 3. model config
    # =========================
    pos_ratio = y.mean()
    neg_ratio = 1.0 - pos_ratio
    scale_pos_weight = neg_ratio / pos_ratio

    models = {
        "hgb": HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_depth=6,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=RANDOM_STATE
        ),
        "xgb": XGBClassifier(
            n_estimators=600,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=2.0,
            min_child_weight=2,
            gamma=0.1,
            objective="binary:logistic",
            eval_metric="aucpr",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist"
        )
    }

    # =========================
    # 4. CV
    # =========================
    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    oof_pred_each_model = {name: np.zeros(len(X_imp)) for name in models}
    test_pred_each_model = {name: np.zeros(len(X_test_imp)) for name in models}
    single_model_results = {}

    for model_name, model in models.items():
        print(f"\n===== Training {model_name} =====")
        fold_scores = []
        fold_thresholds = []

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
            X_tr, X_va = X_imp.iloc[tr_idx], X_imp.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            pos_weight_fold = (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)
            sample_weight = np.where(y_tr == 1, pos_weight_fold, 1.0)

            if model_name == "hgb":
                model.fit(X_tr, y_tr, sample_weight=sample_weight)
            else:
                model.fit(X_tr, y_tr)

            va_prob = model.predict_proba(X_va)[:, 1]
            te_prob = model.predict_proba(X_test_imp)[:, 1]

            oof_pred_each_model[model_name][va_idx] = va_prob
            test_pred_each_model[model_name] += te_prob / N_SPLITS

            best_t, best_f1 = search_best_threshold_two_stage(y_va, va_prob)
            fold_thresholds.append(best_t)
            fold_scores.append(best_f1)

            print(f"Fold {fold}: best threshold = {best_t:.4f}, macro-F1 = {best_f1:.5f}")

        overall_t, overall_f1 = search_best_threshold_two_stage(y, oof_pred_each_model[model_name])

        single_model_results[model_name] = {
            "fold_thresholds": fold_thresholds,
            "fold_scores": fold_scores,
            "oof_best_threshold": overall_t,
            "oof_macro_f1": overall_f1,
            "mean_cv_macro_f1": float(np.mean(fold_scores))
        }

        print(f"[{model_name}] OOF best threshold = {overall_t:.4f}, OOF macro-F1 = {overall_f1:.5f}")
        print(f"[{model_name}] mean CV macro-F1 = {np.mean(fold_scores):.5f}")

    # =========================
    # 5. save single model submissions
    # =========================
    print("\n===== Saving single-model submissions =====")
    for model_name in models.keys():
        best_t = single_model_results[model_name]["oof_best_threshold"]
        best_f1 = single_model_results[model_name]["oof_macro_f1"]

        print(f"{model_name}: OOF best threshold = {best_t:.4f}, OOF macro-F1 = {best_f1:.5f}")
        save_submission_from_prob(
            test_prob=test_pred_each_model[model_name],
            threshold=best_t,
            filename=f"{output_prefix}_{model_name}.csv"
        )

    # =========================
    # 6. search best probability ensemble
    # =========================
    print("\n===== Searching best ensemble weights (probability, hgb + xgb) =====")
    best_weights, best_threshold, best_score = search_best_ensemble_weights(
        y_true=y,
        oof_dict=oof_pred_each_model,
        model_names=["hgb", "xgb"],
        weight_step=WEIGHT_STEP
    )

    print("Best prob weights:", best_weights)
    print("Best prob threshold:", round(best_threshold, 4))
    print("Best prob OOF Macro-F1:", round(best_score, 5))

    test_ensemble = build_ensemble_prediction(test_pred_each_model, best_weights)
    save_submission_from_prob(
        test_prob=test_ensemble,
        threshold=best_threshold,
        filename=f"{output_prefix}_ensemble_prob_hgb_xgb.csv"
    )

    # =========================
    # 7. search best rank ensemble
    # =========================
    print("\n===== Searching best RANK ensemble weights (hgb + xgb) =====")
    best_rank_weights, best_rank_threshold, best_rank_score = search_best_rank_ensemble_weights(
        y_true=y,
        oof_dict=oof_pred_each_model,
        model_names=["hgb", "xgb"],
        weight_step=WEIGHT_STEP
    )

    print("Best rank weights:", best_rank_weights)
    print("Best rank threshold:", round(best_rank_threshold, 4))
    print("Best rank OOF Macro-F1:", round(best_rank_score, 5))

    test_rank_ensemble = build_rank_ensemble_prediction(test_pred_each_model, best_rank_weights)
    save_submission_from_prob(
        test_prob=test_rank_ensemble,
        threshold=best_rank_threshold,
        filename=f"{output_prefix}_ensemble_rank_hgb_xgb.csv"
    )

    # =========================
    # 8. threshold fine report for probability ensemble
    # =========================
    print("\n===== Threshold report: probability ensemble =====")
    oof_ensemble = build_ensemble_prediction(oof_pred_each_model, best_weights)

    prob_candidates = sorted(set([
        round(best_threshold - 0.02, 4),
        round(best_threshold - 0.01, 4),
        round(best_threshold - 0.005, 4),
        round(best_threshold, 4),
        round(best_threshold + 0.005, 4),
        round(best_threshold + 0.01, 4),
        round(best_threshold + 0.02, 4),
    ]))
    prob_candidates = [t for t in prob_candidates if 0.0 <= t <= 1.0]

    rows = []
    for t in prob_candidates:
        score = f1_score(y, (oof_ensemble >= t).astype(int), average="macro")
        rows.append((t, score))

    df_prob = pd.DataFrame(rows, columns=["threshold", "oof_macro_f1"])
    df_prob = df_prob.sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
    print(df_prob)

    topn_prob = min(3, len(df_prob))
    for i in range(topn_prob):
        t = float(df_prob.loc[i, "threshold"])
        pred = (test_ensemble >= t).astype(int)
        pd.DataFrame({
            "Id": np.arange(len(pred)),
            "Bankrupt": pred
        }).to_csv(f"{output_prefix}_ensemble_prob_tune{i+1}.csv", index=False)
        print(f"Saved {output_prefix}_ensemble_prob_tune{i+1}.csv with threshold {t:.4f}")

    # =========================
    # 9. threshold fine report for rank ensemble
    # =========================
    print("\n===== Threshold report: rank ensemble =====")
    oof_rank_ensemble = build_rank_ensemble_prediction(oof_pred_each_model, best_rank_weights)

    rank_candidates = sorted(set([
        round(best_rank_threshold - 0.02, 4),
        round(best_rank_threshold - 0.01, 4),
        round(best_rank_threshold - 0.005, 4),
        round(best_rank_threshold, 4),
        round(best_rank_threshold + 0.005, 4),
        round(best_rank_threshold + 0.01, 4),
        round(best_rank_threshold + 0.02, 4),
    ]))
    rank_candidates = [t for t in rank_candidates if 0.0 <= t <= 1.0]

    rows = []
    for t in rank_candidates:
        score = f1_score(y, (oof_rank_ensemble >= t).astype(int), average="macro")
        rows.append((t, score))

    df_rank = pd.DataFrame(rows, columns=["threshold", "oof_macro_f1"])
    df_rank = df_rank.sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
    print(df_rank)

    topn_rank = min(3, len(df_rank))
    for i in range(topn_rank):
        t = float(df_rank.loc[i, "threshold"])
        pred = (test_rank_ensemble >= t).astype(int)
        pd.DataFrame({
            "Id": np.arange(len(pred)),
            "Bankrupt": pred
        }).to_csv(f"{output_prefix}_ensemble_rank_tune{i+1}.csv", index=False)
        print(f"Saved {output_prefix}_ensemble_rank_tune{i+1}.csv with threshold {t:.4f}")

    # =========================
    # 10. summary
    # =========================
    print("\n===== Summary =====")
    for model_name, result in single_model_results.items():
        print(
            f"{model_name:>4} | "
            f"OOF Macro-F1 = {result['oof_macro_f1']:.5f} | "
            f"Threshold = {result['oof_best_threshold']:.4f}"
        )

    print(
        f" prob2 | OOF Macro-F1 = {best_score:.5f} | "
        f"Threshold = {best_threshold:.4f} | "
        f"Weights = {best_weights}"
    )

    print(
        f" rank2 | OOF Macro-F1 = {best_rank_score:.5f} | "
        f"Threshold = {best_rank_threshold:.4f} | "
        f"Weights = {best_rank_weights}"
    )

    print("\nAll done.\n")

    return {
        "single_model_results": single_model_results,
        "best_prob_weights": best_weights,
        "best_prob_threshold": best_threshold,
        "best_prob_score": best_score,
        "best_rank_weights": best_rank_weights,
        "best_rank_threshold": best_rank_threshold,
        "best_rank_score": best_rank_score,
    }


if __name__ == "__main__":
    run_pipeline(
        clip_low=0.005,
        clip_high=0.995,
        output_prefix="clip005_995"
    )

    # aggressive
    run_pipeline(
        clip_low=0.01,
        clip_high=0.99,
        output_prefix="clip01_99"
    )

    # no clip
    #run_pipeline(
    #    clip_low=0,
    #    clip_high=1,
    #    output_prefix="no_clip"
    #)

Running pipeline with clip_low=0.005, clip_high=0.995
Train shape: (5543, 95)
Test shape : (1386, 95)
Positive ratio: 0.04762763846292621

===== Training hgb =====
Fold 1: best threshold = 0.1440, macro-F1 = 0.77284
Fold 2: best threshold = 0.6400, macro-F1 = 0.84757
Fold 3: best threshold = 0.2500, macro-F1 = 0.80332
Fold 4: best threshold = 0.5400, macro-F1 = 0.82883
Fold 5: best threshold = 0.1400, macro-F1 = 0.79228
[hgb] OOF best threshold = 0.2600, OOF macro-F1 = 0.79157
[hgb] mean CV macro-F1 = 0.80897

===== Training xgb =====
Fold 1: best threshold = 0.4700, macro-F1 = 0.78245
Fold 2: best threshold = 0.4800, macro-F1 = 0.82782
Fold 3: best threshold = 0.6400, macro-F1 = 0.78204
Fold 4: best threshold = 0.7600, macro-F1 = 0.83726
Fold 5: best threshold = 0.2100, macro-F1 = 0.78124
[xgb] OOF best threshold = 0.4650, OOF macro-F1 = 0.78864
[xgb] mean CV macro-F1 = 0.80216

===== Saving single-model submissions =====
hgb: OOF best threshold = 0.2600, OOF macro-F1 = 0.79157
Saved 